# UMUD R45T — rozcięgna v2 + kierunek pęczków R22 (Colab, GPU)

**Jak uruchomić:** GPU → _Środowisko wykonawcze → Uruchom wszystko_. Po rozłączeniu znów _Uruchom wszystko_ — to, co policzone, zostanie pominięte,
trening wznowi się od ostatniego checkpointu. **Zmieniaj tylko komórkę 1 (ustawienia).** Komórka 2 zawiera spakowany kod sprawdzany sumą SHA‑256;
edycja długiego napisu go uszkadza (wtedy notebook spróbuje kopii kodu zapisanej na Drive, a jeśli jej nie ma — poprosi o świeży plik).

**Po co.** Szacunkowo ok. 0,04 wyniku to MT. Model rozcięgien R12B to zamrożony DINOv2 na siatce 37×37 z wejściem 518×518 o zniekształconych proporcjach.
Ten notebook trenuje nowy model rozcięgien w pełnej rozdzielczości (ConvNeXt‑Tiny U‑Net jak R22, 5 foldów) i liczy wszystkie dekodery z nim.
Etap Q (głowica kierunku pęczków R22 + R12B) jest już policzony i zostanie pominięty.

| etap | co robi | czas (T4 / L4) | wynik na Drive |
|---|---|---|---|
| **Q** | 5 modeli R22 z kierunkiem + R12B (gotowe) | ok. 15 min | `outputs/R45T_v1_0_0/Q/` |
| **T** | trening rozcięgien v2, 5 foldów × 16 epok; na T4 w fp16 | ok. 1 h na fold / ok. 30 min | `outputs/R45T_v1_0_0/apo2/` |
| **F** | dekodery z rozcięgnami v2 po 1., 3. i 5. foldzie | ok. 15 min każdy | `outputs/R45T_v1_0_0/F1/`, `F3/`, `F5/` |

Cache danych treningowych budowany jest raz (ok. 20 min odczytu z Drive) i zapisywany na Drive jako jeden plik — kolejne sesje rozpakowują go w ok. minutę.

**Co mi przysłać:** tylko informację, który etap się skończył — pliki odczytam z Drive. Notebook niczego nie zgłasza na Kaggle.
Nie uruchamiaj dwóch sesji na tym samym `RUN_NAME` jednocześnie.


In [ ]:
# =====================================================================================
# KOMÓRKA 1 — USTAWIENIA (tylko tę komórkę można zmieniać)
# =====================================================================================
RUN_NAME = 'R45T_v1_0_0'   # ten sam katalog = wznowienie; nowy = nowy eksperyment
FOLDS = [0, 1, 2, 3, 4]    # foldy rozcięgien v2 do wytrenowania
EXPORT_AFTER = [1, 3, 5]   # eksport F po tylu ukończonych foldach
RUN_QUICK = True           # etap Q (pomijany, jeśli już policzony)
RUN_TRAIN = True           # etap T (+ automatyczne eksporty F)
SESSION_HOURS = 11.0       # po tylu godzinach sesji nie zaczyna nowej epoki (checkpoint zostaje)
CHECK_IMAGE_SHA = True


In [ ]:
# =====================================================================================
# KOMÓRKA 2 — KOD. NIE EDYTUJ TEJ KOMÓRKI: zawiera spakowany kod (sprawdzany sumą SHA-256),
# każda zmiana w długim napisie poniżej go uszkodzi. Ustawienia są w komórce 1.
# =====================================================================================
import os, sys, json, time, base64, zlib, hashlib, importlib.util, subprocess, shutil
from pathlib import Path
T_START = time.time()
FIXTURE = os.environ.get('R45T_FIXTURE_ROOT')          # wyłącznie testy lokalne
if FIXTURE:
    DRIVE_ROOT = Path(FIXTURE) / 'MyDrive'; LOCAL = Path(FIXTURE) / 'content'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive'); LOCAL = Path('/content')
PROJECT = DRIVE_ROOT / 'UMUD_Challenge'
RUN = PROJECT / 'outputs' / RUN_NAME; RUN.mkdir(parents=True, exist_ok=True)
SRC = LOCAL / 'r45t_src'
PACKAGE_SHA = '22cf27dc87d0c2455ee807423b448c84927da418ea4676a908ce6c7931c009ac'
PAYLOAD = '...'

def _unpack(b64):
    raw = zlib.decompress(base64.b64decode(b64))
    if hashlib.sha256(raw).hexdigest() != PACKAGE_SHA:
        raise ValueError('SHA-256 kodu się nie zgadza')
    return raw

_backup = RUN / 'code' / f'r45t_{PACKAGE_SHA[:16]}.b64'
try:
    raw = _unpack(PAYLOAD)
    if not _backup.exists():
        _backup.parent.mkdir(parents=True, exist_ok=True); _backup.write_text(PAYLOAD)
except Exception as _e:
    if _backup.exists():
        raw = _unpack(_backup.read_text())
        print('UWAGA: kod w tej komórce był uszkodzony — użyłem nienaruszonej kopii z Drive:', _backup)
    else:
        raise RuntimeError('Kod w tej komórce jest uszkodzony (zwykle po edycji). Wczytaj świeży plik notebooka '
                           'i zmieniaj tylko komórkę 1 z ustawieniami.') from _e
for rel, content in json.loads(raw).items():
    p = SRC / rel; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(content)
for module, pip_name in [('cv2', 'opencv-python-headless'), ('skimage', 'scikit-image'), ('timm', 'timm')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
sys.path.insert(0, str(SRC))
import numpy as np, pandas as pd, torch, cv2
import importlib as _il
import r45_core, r45_models, r45_decode, r45_apo, r45_infer
for _m in (r45_core, r45_models, r45_decode, r45_apo, r45_infer):   # po ponownym uruchomieniu: zawsze świeży kod
    _il.reload(_m)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu' and not FIXTURE:
    raise RuntimeError('Brak GPU. Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU, potem Uruchom wszystko.')
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True

def log(*a):
    msg = ' '.join(str(x) for x in a); print(msg, flush=True)
    with open(RUN / 'LOG.txt', 'a') as fh:
        fh.write(time.strftime('%Y-%m-%d %H:%M:%S ') + msg + '\n')

def show(df, n=40):
    print(df.head(n).to_string(index=False), flush=True)

_prec = r45_apo.precision_mode(DEVICE)[0]
log('R45T start | GPU:', GPU, '| trening w', str(_prec).replace('torch.', ''), '| torch', torch.__version__,
    '| kod', PACKAGE_SHA[:12], '| RUN', RUN)


In [ ]:
# =====================================================================================
# KOMÓRKA 3 — DANE I CHECKPOINTY (kontrola sum)
# =====================================================================================
ASSETS = SRC / 'assets'
R22RUN = PROJECT / 'outputs' / 'R22_v1_0_0'
R12B_CKPT = DRIVE_ROOT / 'UMUD_R12B' / 'best_dinov2_frozen.pt'
R12B_SHA = os.environ.get('R45T_R12B_SHA', '21ec34d0f5dd7664ecab0ca608f976fdf9d2a161e2af2873c21d513cd5ae19ef')
SIDE = int(os.environ.get('R45T_SIDE', 768))
test = r45_infer.load_manifest(Path(os.environ.get('R45T_TEST_MANIFEST', ASSETS / 'test_manifest.csv')), PROJECT, CHECK_IMAGE_SHA)
osf = r45_infer.load_manifest(Path(os.environ.get('R45T_OSF_MANIFEST', ASSETS / 'osf_manifest.csv')), PROJECT, CHECK_IMAGE_SHA)
log('obrazy: test', len(test), '| OSF', len(osf))

tc = json.loads((R22RUN / 'TRAINING_COMPLETE.json').read_text())
ver_path = RUN / 'R22_VERIFIED.json'
ver = json.loads(ver_path.read_text()) if ver_path.exists() else {}
R22_PATHS = []
for d in sorted(tc['checkpoints'], key=lambda d: d['fold']):
    p = R22RUN / f"fold_{d['fold']}" / 'final.pt'
    key = f'{p}|{p.stat().st_size}|{int(p.stat().st_mtime)}'
    if ver.get(key) != d['checkpoint_sha256']:
        assert r45_core.sha256(p) == d['checkpoint_sha256'], f"R22 fold {d['fold']}: checkpoint różni się od TRAINING_COMPLETE.json"
        ver[key] = d['checkpoint_sha256']
    R22_PATHS.append(p)
r45_core.atomic_json(ver_path, ver)
assert len(R22_PATHS) == 5, 'Potrzebne 5 foldów R22'
assert r45_core.sha256(R12B_CKPT) == R12B_SHA, 'Inny checkpoint rozcięgien R12B'

PRETRAINED = R22RUN / 'pretrained' / 'convnext_tiny_imagenet1k_v1_encoder.pt'
if not PRETRAINED.exists():
    from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
    PRETRAINED = RUN / 'pretrained' / 'convnext_tiny_imagenet1k_v1_encoder.pt'
    if not PRETRAINED.exists():
        torch.hub.set_dir(str(RUN / 'pretrained' / 'torch_hub'))
        m = convnext_tiny(weights=None); m.load_state_dict(ConvNeXt_Tiny_Weights.IMAGENET1K_V1.get_state_dict(progress=True, check_hash=True))
        r45_models.atomic_torch(PRETRAINED, {'encoder': m.features.state_dict()})
SIG_BASE = {'package': PACKAGE_SHA, 'r22': [str(p) for p in R22_PATHS], 'side': SIDE, 'tta': True}
OUTQ = RUN / 'Q'
APO_DIR = RUN / 'apo2'; APO_DIR.mkdir(exist_ok=True)
log('checkpointy R22 (5) i R12B zgodne; enkoder ImageNet:', PRETRAINED)


## Etap Q — kierunek pęczków R22 + rozcięgna R12B (ok. 20–30 min)

Kontrola odtworzenia: dekodery liniowe liczone tu z R12B muszą zgadzać się z zapisanymi pomiarami R22 (tabela `REPRO`).


In [ ]:
# =====================================================================================
# KOMÓRKA 4 — ETAP Q
# =====================================================================================
def load_r22_models():
    return [r45_models.load_r22(p, DEVICE)[0] for p in R22_PATHS]

def cuda_gc():
    import gc
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

TRUTH = r45_infer.osf_truth(ASSETS / 'osf35_full.csv')
SAVED = pd.read_csv(ASSETS / 'r22_saved_measurements.csv')

def reports(name, frame_test, frame_osf, out):
    tab = r45_infer.osf_table(frame_osf, TRUTH); tab.to_csv(out / f'{name}_OSF_TABLE.csv', index=False)
    rt = r45_infer.repro_table(frame_test, SAVED[SAVED.set == 'test'])
    ro = r45_infer.repro_table(frame_osf, SAVED[SAVED.set == 'osf'])
    rt.to_csv(out / f'{name}_REPRO_test.csv', index=False); ro.to_csv(out / f'{name}_REPRO_osf.csv', index=False)
    print('\nOSF-35 (wzgledem sredniej 7 ekspertow), najlepsze kolumny per target:')
    show(tab.groupby('target').head(6))
    print('\nREPRO (dekodery z R12B vs zapisane R22), test:')
    show(rt)
    return tab, rt, ro

if RUN_QUICK:
    if (OUTQ / 'Q_SUMMARY.json').exists():
        log('Etap Q: już gotowy (Q/Q_test.csv, Q/Q_osf.csv)')
    else:
        r22 = load_r22_models(); r12b = r45_models.load_r12b(R12B_CKPT, DEVICE)
        src = {'r12b': lambda path, gray, square: r45_models.predict_r12b(r12b, path, DEVICE)}
        sig = {**SIG_BASE, 'stage': 'Q', 'r12b': R12B_SHA}
        q_test = r45_infer.run_stage('Q_test', test, OUTQ, r22, src, sig, SIDE, DEVICE, log=log)
        q_osf = r45_infer.run_stage('Q_osf', osf, OUTQ, r22, src, sig, SIDE, DEVICE, log=log)
        tab, rt, ro = reports('Q', q_test, q_osf, OUTQ)
        r45_core.atomic_json(OUTQ / 'Q_SUMMARY.json', r45_core.clean_json({
            'gpu': GPU, 'minutes': (time.time() - T_START) / 60, 'test_rows': len(q_test), 'osf_rows': len(q_osf),
            'decode_errors': int((q_test.decode_status != 'OK').sum() + (q_osf.decode_status != 'OK').sum()),
            'repro_test': rt.to_dict('records'), 'osf_best': tab.groupby('target').head(5).to_dict('records')}))
        del r22, r12b, src; cuda_gc()
        log('ETAP Q GOTOWY ->', OUTQ)


## Etap T — trening rozcięgien v2 (wznawialny)

5 foldów po grupach kolejnych klatek, stała liczba epok (bez wyboru na OSF/LB). Checkpoint co 2 epoki; po rozłączeniu
uruchom wszystko ponownie — ukończone foldy i etap Q zostaną pominięte. Eksporty F po 1., 3. i 5. foldzie.


In [ ]:
# =====================================================================================
# KOMÓRKA 5 — ETAP T (trening) + automatyczne eksporty F
# =====================================================================================
CFG = dict(r45_apo.APO_CONFIG)
if FIXTURE:
    CFG.update(json.loads(os.environ.get('R45T_APO_OVERRIDES', '{}')))
assert CFG['side'] == SIDE, 'Rozdzielczość rozcięgien v2 musi równać się SIDE (wspólny letterbox)'
TRAIN_ROOT = PROJECT / 'data' / 'competition_train_v1'

def completed_folds():
    return sorted(int(p.parent.name.split('_')[1]) for p in APO_DIR.glob('fold_*/DONE.json'))

def run_final(k):
    folds = completed_folds()
    if len(folds) < k:
        return None
    out = RUN / f'F{k}'
    if (out / f'F{k}_SUMMARY.json').exists():
        log(f'F{k}: już policzony'); return out
    done = [json.loads((APO_DIR / f'fold_{f}' / 'DONE.json').read_text()) for f in folds]
    apo_models = [r45_models.load_apo2(APO_DIR / f'fold_{f}' / 'final.pt', DEVICE)[0] for f in folds]
    r22 = load_r22_models()
    src = {'v2': lambda path, gray, square: r45_core.restore_map(r45_models.predict_apo2(apo_models, square, DEVICE, True), gray.shape)}
    sig = {**SIG_BASE, 'stage': f'F{k}', 'apo2': [d['final_sha256'] for d in done]}
    f_test = r45_infer.run_stage(f'F{k}_test', test, out, r22, src, sig, SIDE, DEVICE, log=log)
    f_osf = r45_infer.run_stage(f'F{k}_osf', osf, out, r22, src, sig, SIDE, DEVICE, log=log)
    del r22, apo_models, src; cuda_gc()
    merged = {}
    for name, f in (('test', f_test), ('osf', f_osf)):
        f = f.rename(columns={'decode_status': 'v2_decode_status', 'seconds': 'v2_seconds'})
        qp = OUTQ / f'Q_{name}.csv'
        if qp.exists():
            q = pd.read_csv(qp)
            keep = [c for c in f.columns if c == 'image_id' or c not in q.columns]
            f = q.merge(f[keep], on='image_id', how='outer', validate='one_to_one')
        f.to_csv(out / f'R45T_F{k}_{name}.csv', index=False); merged[name] = f
        shutil.copyfile(out / f'R45T_F{k}_{name}.csv', RUN / f'R45T_LATEST_{name}.csv')
    tab, rt, ro = reports(f'F{k}', merged['test'], merged['osf'], out)
    r45_core.atomic_json(out / f'F{k}_SUMMARY.json', r45_core.clean_json({
        'folds': folds, 'gpu': GPU, 'oof': [d['oof_summary'] for d in done],
        'osf_best': tab.groupby('target').head(8).to_dict('records')}))
    log(f'EKSPORT F{k} GOTOWY ->', out)
    return out

if RUN_TRAIN:
    pairs, prep = r45_apo.list_pairs(TRAIN_ROOT / 'apo_images', TRAIN_ROOT / 'apo_masks')
    log('pary obraz+maska rozcięgien:', prep['pairs'], '| bez maski:', len(prep['images_without_mask']))
    frame = r45_apo.assign_folds(pairs, CFG['group_size'], CFG['n_folds'], CFG['seed'])
    fp = APO_DIR / 'FOLDS.csv'
    if fp.exists():
        old = pd.read_csv(fp)
        same = (old.name.astype(str).tolist() == frame.name.astype(str).tolist() and
                old.fold.astype(int).tolist() == frame.fold.astype(int).tolist() and
                old.group_id.astype(int).tolist() == frame.group_id.astype(int).tolist())
        assert same, 'Inny podział niż w poprzedniej sesji (zmienione dane?) — użyj nowego RUN_NAME'
    else:
        frame.to_csv(fp, index=False)
    CACHE = LOCAL / f'r45t_apo_cache_{CFG["side"]}'
    cidx = r45_apo.sync_cache(frame, CACHE, APO_DIR / f'cache_{CFG["side"]}.tar', CFG['side'], log=log)
    log('cache rozcięgien:', len(cidx), 'obrazów | niezgodny rozmiar maski:', int(cidx.shape_mismatch.sum()),
        '| średni udział maski:', round(float(cidx.fg_frac.mean()), 4))
    bp = APO_DIR / 'BATCH.json'
    if bp.exists():
        b = json.loads(bp.read_text())
    else:
        bb, aa = r45_apo.pick_batch(PRETRAINED, CFG, DEVICE); b = {'batch': bb, 'accumulation': aa, 'gpu': GPU}
        r45_core.atomic_json(bp, b)
    CFG['_batch'], CFG['_accumulation'] = b['batch'], b['accumulation']
    log('batch', b['batch'], 'x akumulacja', b['accumulation'], '| epoki', CFG['epochs'], '| foldy', FOLDS)
    deadline = T_START + SESSION_HOURS * 3600
    budget_hit = False
    for fold in FOLDS:
        while True:
            oom = False
            try:
                d = r45_apo.train_fold(frame, CACHE, APO_DIR, PRETRAINED, CFG, fold, DEVICE, log=log, deadline=deadline)
                log(f'fold {fold} OOF:', json.dumps(d['oof_summary'])); break
            except r45_apo.SessionBudgetReached as e:
                log(str(e), '— uruchom wszystko ponownie w nowej sesji'); budget_hit = True; break
            except torch.cuda.OutOfMemoryError:
                if CFG['_batch'] == 1:
                    raise
                oom = True
            if oom:  # outside the except block, so the failed step's tensors can be freed
                cuda_gc()
                CFG['_accumulation'] *= CFG['_batch']; CFG['_batch'] = 1
                r45_core.atomic_json(bp, {'batch': 1, 'accumulation': CFG['_accumulation'], 'gpu': GPU, 'after_oom': True})
                log('Brak pamięci GPU przy batch 2 — przechodzę na batch 1 i wznawiam ten fold od checkpointu')
        if budget_hit:
            break
        k = len(completed_folds())
        if k in EXPORT_AFTER:
            run_final(k)
    rows = []
    for f in completed_folds():
        rows.append({'fold': f, **json.loads((APO_DIR / f'fold_{f}' / 'DONE.json').read_text())['oof_summary']})
    if rows:
        pd.DataFrame(rows).to_csv(APO_DIR / 'OOF_SUMMARY.csv', index=False); show(pd.DataFrame(rows))
    log('Ukończone foldy:', completed_folds(), '| brakujące:', [f for f in FOLDS if f not in completed_folds()])


## Etap F — eksport z wszystkimi ukończonymi foldami

Jeżeli trening skończył się w tej sesji, eksport F5 jest już policzony i ta komórka nic nie zrobi.
Pliki do analizy: `R45T_LATEST_test.csv`, `R45T_LATEST_osf.csv` oraz tabele `*_OSF_TABLE.csv`.


In [ ]:
# =====================================================================================
# KOMÓRKA 6 — ETAP F (z ukończonymi foldami) i podsumowanie
# =====================================================================================
k = len(completed_folds())
if k and not (RUN / f'F{k}' / f'F{k}_SUMMARY.json').exists():
    run_final(k)
files = sorted(str(p.relative_to(RUN)) for p in RUN.rglob('*') if p.suffix in ('.csv', '.json') and 'cache' not in str(p))
print('\n'.join(files))
log(f'GOTOWE. Ukończone foldy rozcięgien v2: {k}/5. Daj znać, który etap się skończył — pliki odczytam z Drive ({RUN}).')
